## Data Preprocessing

## Imports

In [5]:
import kagglehub
from pathlib import Path
import pandas as pd

## Download the Data from Kaggle

In [ ]:
path = kagglehub.dataset_download(
    "samartalwar/nvidia-360-stock-cloud-ux-and-semiconductor-macro",
    output_dir="../data"
)

print("Path to dataset files:", path)

100%|██████████| 1.20M/1.20M [00:00<00:00, 5.64MB/s]

Extracting files...
Path to dataset files: ../data


## Initial Preview of the Datasets

There are two. Note the shape (observations x features). Let's us see all the features (columns).

In [6]:
DATA_DIR = Path("../data")

reviews = pd.read_csv(DATA_DIR / "geforcenow_app_reviews_raw.csv")
market = pd.read_csv(DATA_DIR / "nvidia_daily_master_360.csv")

print("REVIEWS")
print("Shape:", reviews.shape)
display(reviews.head())

print("\nMARKET DATA")
print("Shape:", market.shape)
display(market.head())

REVIEWS
Shape: (12008, 8)


,review_id,author_name,review_text,rating,thumbs_up,posted_at,sentiment_score,sentiment_category
0,643644d0-54d3-4ad4-bd08-aae9ca99cbed,Micale Clibe,One of the best cloud gaming app,5,0,2026-09-19 09:19:56,0.6369,Positive
1,1a3b3411-cb2b-47c9-bcc9-0b19769dd8db,Michele Zhou,this app is amazing tho unfortunately sometime...,4,0,2026-09-19 08:05:53,0.8205,Positive
2,286832d8-3ae0-4bf4-8b0f-73c542f71e71,Mahesh Bharmal,useless app no free membership available,1,0,2026-09-19 04:17:54,-0.1779,Negative
3,bd88fd97-05a7-4799-91e4-551371cd7ee7,Angelina Breanna,doesn't login. waste of time and money. fix th...,1,0,2026-09-18 19:38:12,0.3252,Positive
4,9746b622-31fe-47ea-8be5-b4e3c0926cbf,Nikhil Gowala,very bad app UI,1,0,2026-09-18 16:59:00,-0.5849,Negative



MARKET DATA
Shape: (4280, 14)


,date,nvda_close,nvda_volume,tsm_close,smh_etf_close,qqq_close,nvda_return_pct,nvda_volatility_30d,nvda_sma_50,nvda_sma_200,is_trading_day,gfn_reviews_count,gfn_avg_rating,gfn_sentiment_score
0,2015-01-01,NaN,0,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,0,0,NaN,NaN
1,2015-01-02,0.50325,113680000,22.280001,27.245001,102.940002,0.000000,NaN,NaN,NaN,1,0,NaN,NaN
2,2015-01-03,0.50325,0,22.280001,27.245001,102.940002,0.000000,NaN,NaN,NaN,0,0,NaN,NaN
3,2015-01-04,0.50325,0,22.280001,27.245001,102.940002,0.000000,NaN,NaN,NaN,0,0,NaN,NaN
4,2015-01-05,0.49475,197952000,21.740000,26.760000,101.430000,-1.689023,NaN,NaN,NaN,1,0,NaN,NaN


## Check for Duplicates and Missing Values

In [7]:
# Basic dataset inspection

print("=== REVIEW DATA ===")
print(reviews.info())

print("\nMissing values:")
display(reviews.isna().sum().to_frame("missing"))

print("Duplicate review IDs:", reviews["review_id"].duplicated().sum())


print("\n=== MARKET DATA ===")
print(market.info())

print("\nMissing values:")
display(market.isna().sum().to_frame("missing"))

print("Duplicate dates:", market["date"].duplicated().sum())

=== REVIEW DATA ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12008 entries, 0 to 12007
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   review_id           12008 non-null  object 
 1   author_name         12008 non-null  object 
 2   review_text         12008 non-null  object 
 3   rating              12008 non-null  int64  
 4   thumbs_up           12008 non-null  int64  
 5   posted_at           12008 non-null  object 
 6   sentiment_score     12008 non-null  float64
 7   sentiment_category  12008 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 750.6+ KB
None

Missing values:


,missing
review_id,0
author_name,0
review_text,0
rating,0
thumbs_up,0
posted_at,0
sentiment_score,0
sentiment_category,0


Duplicate review IDs: 0

=== MARKET DATA ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4280 entries, 0 to 4279
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   date                 4280 non-null   object 
 1   nvda_close           4279 non-null   float64
 2   nvda_volume          4280 non-null   int64  
 3   tsm_close            4279 non-null   float64
 4   smh_etf_close        4279 non-null   float64
 5   qqq_close            4279 non-null   float64
 6   nvda_return_pct      4280 non-null   float64
 7   nvda_volatility_30d  4233 non-null   float64
 8   nvda_sma_50          4206 non-null   float64
 9   nvda_sma_200         3992 non-null   float64
 10  is_trading_day       4280 non-null   int64  
 11  gfn_reviews_count    4280 non-null   int64  
 12  gfn_avg_rating       2268 non-null   float64
 13  gfn_sentiment_score  2268 non-null   float64
dtypes: float64(10), int64(3), object(1)
memory 

,missing
date,0
nvda_close,1
nvda_volume,0
tsm_close,1
smh_etf_close,1
qqq_close,1
nvda_return_pct,0
nvda_volatility_30d,47
nvda_sma_50,74
nvda_sma_200,288


Duplicate dates: 0


## Some final basic preprocessing

In [8]:
# ----------------------------
# Review preprocessing
# ----------------------------

reviews["posted_at"] = pd.to_datetime(
    reviews["posted_at"],
    errors="coerce"
)

# Create a daily date for joining to market data
reviews["date"] = reviews["posted_at"].dt.normalize()

# Remove duplicate reviews if any appear
reviews = reviews.drop_duplicates(subset="review_id")

# Clean text whitespace
reviews["review_text"] = (
    reviews["review_text"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)


# ----------------------------
# Market preprocessing
# ----------------------------

market["date"] = pd.to_datetime(
    market["date"],
    errors="coerce"
)

market = (
    market
    .drop_duplicates(subset="date")
    .sort_values("date")
    .reset_index(drop=True)
)


print(
    "Review date range:",
    reviews["date"].min(),
    "to",
    reviews["date"].max()
)

print(
    "Market date range:",
    market["date"].min(),
    "to",
    market["date"].max()
)

Review date range: 2020-05-24 00:00:00 to 2026-09-19 00:00:00
Market date range: 2015-01-01 00:00:00 to 2026-09-19 00:00:00


In [9]:
daily_reviews = (
    reviews
    .groupby("date")
    .agg(
        gfn_reviews_count=("review_id", "count"),
        gfn_avg_rating=("rating", "mean"),
        gfn_sentiment_score=("sentiment_score", "mean"),
        gfn_avg_thumbs_up=("thumbs_up", "mean"),

        gfn_one_star_share=(
            "rating",
            lambda x: (x == 1).mean()
        ),

        gfn_negative_share=(
            "sentiment_category",
            lambda x: (x == "Negative").mean()
        ),

        gfn_positive_share=(
            "sentiment_category",
            lambda x: (x == "Positive").mean()
        )
    )
    .reset_index()
)

display(daily_reviews.head(10))

,date,gfn_reviews_count,gfn_avg_rating,gfn_sentiment_score,gfn_avg_thumbs_up,gfn_one_star_share,gfn_negative_share,gfn_positive_share
0,2020-05-24,4,3.250000,-0.035900,0.000000,0.250000,0.500000,0.500000
1,2020-05-25,2,4.500000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2020-05-26,5,4.000000,0.574640,1.200000,0.000000,0.000000,1.000000
3,2020-05-27,2,1.500000,-0.088950,0.000000,0.500000,0.500000,0.000000
4,2020-05-28,6,4.333333,0.734067,7.333333,0.166667,0.000000,0.833333
5,2020-05-29,4,5.000000,0.617475,0.000000,0.000000,0.000000,1.000000
6,2020-05-30,5,3.200000,0.375320,13.600000,0.000000,0.400000,0.600000
7,2020-05-31,4,4.250000,0.350425,0.750000,0.000000,0.250000,0.750000
8,2020-06-01,6,4.500000,0.403517,0.333333,0.000000,0.166667,0.833333
9,2020-06-02,2,1.000000,0.074350,3.500000,1.000000,0.500000,0.500000


In [10]:
# Remove the dataset creator's pre-aggregated review columns
# because we are rebuilding them directly from the raw reviews.

gfn_existing_cols = [
    "gfn_reviews_count",
    "gfn_avg_rating",
    "gfn_sentiment_score"
]

market_clean = market.drop(
    columns=gfn_existing_cols,
    errors="ignore"
)

# Merge market data with our daily review features
combined = market_clean.merge(
    daily_reviews,
    on="date",
    how="left"
)

# No reviews on a day means review count = 0
combined["gfn_reviews_count"] = (
    combined["gfn_reviews_count"]
    .fillna(0)
    .astype(int)
)

display(
    combined[
        [
            "date",
            "nvda_close",
            "nvda_return_pct",
            "gfn_reviews_count",
            "gfn_avg_rating",
            "gfn_sentiment_score",
            "gfn_negative_share",
            "gfn_one_star_share"
        ]
    ].tail(20)
)

,date,nvda_close,nvda_return_pct,gfn_reviews_count,gfn_avg_rating,gfn_sentiment_score,gfn_negative_share,gfn_one_star_share
4260,2026-08-31,220.779999,1.484714,6,2.833333,0.135117,0.333333,0.500000
4261,2026-09-01,217.440002,-1.512817,3,1.000000,0.290200,0.000000,1.000000
4262,2026-09-02,224.410004,3.205482,7,3.714286,-0.030286,0.285714,0.142857
4263,2026-09-03,228.449997,1.800273,2,1.000000,0.449600,0.000000,1.000000
4264,2026-09-04,230.360001,0.836071,6,3.000000,0.131467,0.333333,0.500000
4265,2026-09-05,230.360001,0.000000,10,3.500000,0.363660,0.100000,0.200000
4266,2026-09-06,230.360001,0.000000,9,3.666667,0.212411,0.333333,0.333333
4267,2026-09-07,230.360001,0.000000,4,2.750000,0.124725,0.500000,0.500000
4268,2026-09-08,225.729996,-2.009900,3,2.333333,-0.632600,1.000000,0.666667
4269,2026-09-09,223.669998,-0.912594,4,2.500000,0.336775,0.250000,0.250000


## Save the combined dataset for use in the other notebooks

In [11]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

combined.to_csv(
    PROCESSED_DIR / "nvidia_gfn_combined.csv",
    index=False
)

print("Saved combined dataset.")

Saved combined dataset.
